# DJI Tello Final Experiment Analysis

This notebook is intentionally reproducible: every figure used for the analysis is generated by code in the notebook and saved to `notebooks/images/`.

The analysis uses the final experiment CSV files from `results/` and avoids `pandas` so it can run with only Python, `numpy`, and `matplotlib`.


## Setup And Data Loading

The first cells locate the project root, load every metrics/state CSV, and define helper functions used by all plots.


In [ ]:
from pathlib import Path
import csv
import math
import os
import statistics
from collections import OrderedDict

os.environ.setdefault("MPLCONFIGDIR", "/tmp/tello_matplotlib_cache")

import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
if (cwd / "results").exists() and (cwd / "notebooks").exists():
    ROOT = cwd
elif (cwd.parent / "results").exists() and cwd.name == "notebooks":
    ROOT = cwd.parent
else:
    ROOT = Path("..").resolve()

RESULTS_DIR = ROOT / "results"
IMAGES_DIR = ROOT / "notebooks" / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

QUALITY_COLORS = {
    "OK": "#4C9F70",
    "DEGRADED": "#E6A23C",
    "STALE": "#D95F5F",
    "NO_DATA": "#8A8F98",
    "": "#8A8F98",
}

QUALITY_EXPLANATION = (
    "Quality score is a freshness/continuity indicator: "
    "OK=100 when latest age <= 200 ms and interarrival <= 300 ms; "
    "DEGRADED=60 when latest age <= 500 ms and interarrival <= 500 ms; "
    "STALE=0 when data is older; NO_DATA=0 when no packet/frame has arrived."
)


def load_csv(path):
    with path.open(newline="") as f:
        return list(csv.DictReader(f))


def to_float(value, default=np.nan):
    if value is None or value == "":
        return default
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def to_int(value, default=0):
    x = to_float(value, np.nan)
    return default if math.isnan(x) else int(x)


def series(rows, column, *, nonnegative=False, nonzero=False):
    values = []
    for row in rows:
        x = to_float(row.get(column))
        if math.isnan(x):
            continue
        if nonnegative and x < 0:
            continue
        if nonzero and x == 0:
            continue
        values.append(x)
    return np.array(values, dtype=float)


def elapsed_seconds(rows, column="elapsed_ms"):
    values = []
    for i, row in enumerate(rows):
        x = to_float(row.get(column))
        if math.isnan(x):
            x = to_float(row.get("steady_elapsed_ms"))
        if math.isnan(x):
            x = to_float(row.get("recording_elapsed_ms"))
        values.append((x / 1000.0) if not math.isnan(x) else float(i))
    return np.array(values, dtype=float)


def xy_series(rows, column, *, time_column="elapsed_ms", nonnegative=False, nonzero=False, max_s=None):
    xs = []
    ys = []
    for i, row in enumerate(rows):
        y = to_float(row.get(column))
        if math.isnan(y):
            continue
        if nonnegative and y < 0:
            continue
        if nonzero and y == 0:
            continue
        x = to_float(row.get(time_column))
        if math.isnan(x):
            x = to_float(row.get("steady_elapsed_ms"))
        if math.isnan(x):
            x = to_float(row.get("recording_elapsed_ms"))
        x = (x / 1000.0) if not math.isnan(x) else float(i)
        if max_s is not None and x > max_s:
            continue
        xs.append(x)
        ys.append(y)
    return np.array(xs, dtype=float), np.array(ys, dtype=float)


def final_value(rows, column, default=np.nan):
    for row in reversed(rows):
        x = to_float(row.get(column), np.nan)
        if not math.isnan(x):
            return x
    return default


def final_text(rows, column, default=""):
    for row in reversed(rows):
        value = row.get(column, "")
        if value != "":
            return value
    return default


def experiment_label(name):
    stem = name.replace("-state", "")
    parts = stem.split("-")
    if parts and parts[-1].isdigit():
        stem = "-".join(parts[:-1])
    return stem


def savefig(filename):
    path = IMAGES_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"saved: {path.relative_to(ROOT)}")


def get_rows(name):
    if name not in metrics:
        raise KeyError(f"missing metrics file: {name}")
    return metrics[name]


def get_state(name):
    if name not in states:
        raise KeyError(f"missing state file: {name}")
    return states[name]


def command_event_time(rows, command):
    for row in rows:
        if row.get("event") == "command:manual" and row.get("last_command") == command:
            return to_float(row.get("elapsed_ms")) / 1000.0
    return np.nan


def add_quality_bands(ax, y_min=-5, y_max=105):
    ax.axhspan(80, y_max, color="#4C9F70", alpha=0.08, label="OK region")
    ax.axhspan(30, 80, color="#E6A23C", alpha=0.08, label="DEGRADED region")
    ax.axhspan(y_min, 30, color="#D95F5F", alpha=0.06, label="STALE/NO_DATA region")
    ax.axhline(100, color="#4C9F70", linestyle=":", linewidth=1)
    ax.axhline(60, color="#E6A23C", linestyle=":", linewidth=1)
    ax.axhline(0, color="#D95F5F", linestyle=":", linewidth=1)
    ax.set_ylim(y_min, y_max)


def physical_flight_markers(state_rows):
    points = []
    for row in state_rows:
        t = to_float(row.get("steady_elapsed_ms")) / 1000.0
        h = to_float(row.get("h"))
        tof = to_float(row.get("tof"))
        vgz = to_float(row.get("vgz"), 0)
        if not math.isnan(t):
            points.append((t, h, tof, vgz))
    takeoff_s = np.nan
    landing_s = np.nan
    touchdown_s = np.nan
    for t, h, tof, vgz in points:
        if t < 5:
            continue
        if (not math.isnan(h) and h >= 10) or (not math.isnan(tof) and tof >= 18):
            takeoff_s = t
            break
    if not math.isnan(takeoff_s):
        for i, (t, h, tof, vgz) in enumerate(points):
            if t < takeoff_s + 30:
                continue
            if math.isnan(h) or math.isnan(tof):
                continue
            if h <= 10 and tof <= 20:
                touchdown_s = t
                break
        for i, (t, h, tof, vgz) in enumerate(points):
            if math.isnan(touchdown_s) or t < max(takeoff_s + 30, touchdown_s - 8):
                continue
            if t > touchdown_s:
                break
            future = points[i:min(len(points), i + 45)]
            reaches_ground = any((not math.isnan(fh) and not math.isnan(ftof) and fh <= 20 and ftof <= 25) for _, fh, ftof, _ in future)
            if reaches_ground and vgz >= 1 and ((not math.isnan(h) and h >= 50) or (not math.isnan(tof) and tof >= 50)):
                landing_s = t
                break
    return takeoff_s, landing_s, touchdown_s


def annotate_takeoff_land(ax, takeoff_s, land_s):
    if not math.isnan(takeoff_s):
        ax.axvline(takeoff_s, color="#4C9F70", linestyle="--", linewidth=1.5)
        ax.text(takeoff_s, 0.98, "takeoff start", transform=ax.get_xaxis_transform(), rotation=90, va="top", ha="right", color="#2F7D4F")
    if not math.isnan(land_s):
        ax.axvline(land_s, color="#D95F5F", linestyle="--", linewidth=1.5)
        ax.text(land_s, 0.98, "landing start", transform=ax.get_xaxis_transform(), rotation=90, va="top", ha="right", color="#A33A3A")

metrics = OrderedDict()
states = OrderedDict()
for path in sorted(RESULTS_DIR.glob("*.csv")):
    rows = load_csv(path)
    if path.stem.endswith("-state"):
        states[path.stem] = rows
    else:
        metrics[path.stem] = rows

print(f"Loaded {len(metrics)} metrics CSV files and {len(states)} state CSV files.")
for name, rows in metrics.items():
    print(f"metrics: {name:28s} rows={len(rows)}")
for name, rows in states.items():
    print(f"state:   {name:28s} rows={len(rows)}")
print(QUALITY_EXPLANATION)


## Data Coverage

This plot is a quick sanity check. It shows which experiment files are present and how many rows each one contains. It is useful before interpreting results because an empty or very short file may indicate that an experiment did not actually run.


In [ ]:
labels = []
counts = []
for name, rows in list(metrics.items()) + list(states.items()):
    labels.append(name)
    counts.append(len(rows))

plt.figure(figsize=(11, max(4, len(labels) * 0.35)))
y = np.arange(len(labels))
plt.barh(y, counts, color="#4C78A8")
plt.yticks(y, labels)
plt.xlabel("rows")
plt.title("Experiment data availability")
for yi, c in zip(y, counts):
    plt.text(c + max(counts) * 0.01, yi, str(c), va="center")
savefig("experiment_data_availability.png")


## Run Summary

This table extracts the final row of each metrics file. It gives a compact view of duration, command failures, telemetry/video freshness labels, and final decode FPS. It is not a substitute for the detailed plots; it is only a run-level index.


In [ ]:
columns = ["experiment", "rows", "duration_s", "cmd_failures", "telemetry", "video", "decode_fps"]
summary = []
for name, rows in metrics.items():
    duration_s = final_value(rows, "elapsed_ms", 0) / 1000.0
    summary.append([
        experiment_label(name),
        len(rows),
        f"{duration_s:.1f}",
        str(to_int(rows[-1].get("command_failures"), 0)) if rows else "0",
        final_text(rows, "telemetry_quality", ""),
        final_text(rows, "video_quality", ""),
        f"{final_value(rows, 'decoder_fps_ema', 0):.1f}",
    ])

widths = [max(len(str(row[i])) for row in [columns] + summary) for i in range(len(columns))]
fmt = " | ".join("{" + str(i) + ":" + str(widths[i]) + "}" for i in range(len(columns)))
print(fmt.format(*columns))
print("-+-".join("-" * w for w in widths))
for row in summary:
    print(fmt.format(*row))


## Command Channel Evidence

The E2 plot uses `command_latency_ms_avg`, which is a cumulative running average. The first point is high because it is based on only one `battery?` response, around 117 ms. Later `battery?` responses were mostly around 48-53 ms, so the running average naturally falls as more samples are added. This is consistent with first-command/warm-up overhead plus cumulative-average convergence.


In [ ]:
rows = get_rows("E2-CMD-BASE-001")
x_avg, y_avg = xy_series(rows, "command_latency_ms_avg", nonnegative=True)

plt.figure(figsize=(10, 5))
plt.plot(x_avg, y_avg, color="#4C78A8", linewidth=2, label="running average latency")
if len(y_avg):
    plt.scatter([x_avg[0]], [y_avg[0]], color="#E45756", zorder=5, label=f"first sample = {y_avg[0]:.0f} ms")
    plt.axhline(y_avg[-1], color="#333333", linestyle="--", label=f"final avg = {y_avg[-1]:.1f} ms")
plt.xlabel("elapsed time (s)")
plt.ylabel("latency (ms)")
plt.title("E2 command latency baseline: cumulative average convergence")
plt.legend()
savefig("e2_command_latency.png")


## Telemetry Freshness

Telemetry freshness is determined from two timing values: latest packet age and packet interarrival time. The score is not a semantic correctness score; it says whether telemetry is fresh enough to trust for monitoring or future feedback control.

- `packet age` is how old the most recently received telemetry packet is at the moment the metrics row is written. If no new packet arrives, age grows.
- `packet interarrival` is the time gap between consecutive telemetry packets. It shows whether packets are arriving regularly.

A run can have low interarrival while packets are flowing, then high age after packets stop. For telemetry to be `OK`, both the latest age and interarrival constraints must be satisfied.

Quality thresholds used by the project:

- `OK = 100`: latest telemetry age <= 200 ms and interarrival <= 300 ms.
- `DEGRADED = 60`: latest telemetry age <= 500 ms and interarrival <= 500 ms.
- `STALE = 0`: data is older than those limits.
- `NO_DATA = 0`: no telemetry packet was received.


The boxplot keeps the distribution view. It answers: how much did telemetry age vary during each run? Lower is better; values near or above 500 ms mean the current state is too old for feedback assumptions.


In [ ]:
labels = []
values = []
for name, rows in metrics.items():
    ages = series(rows, "telemetry_age_ms", nonnegative=True)
    if len(ages):
        labels.append(experiment_label(name))
        values.append(ages)

plt.figure(figsize=(11, 5))
plt.boxplot(values, tick_labels=labels, showfliers=False)
plt.axhline(200, color="#4C9F70", linestyle="--", linewidth=1, label="OK age threshold: 200 ms")
plt.axhline(500, color="#D95F5F", linestyle="--", linewidth=1, label="DEGRADED age limit: 500 ms")
plt.xticks(rotation=35, ha="right")
plt.ylabel("telemetry packet age (ms)")
plt.title("Telemetry age distribution by experiment")
plt.legend()
savefig("telemetry_age_comparison.png")


The timeline view makes the freshness rule visible over time. The first two subplots compare normal runs up to 150 seconds. E8 power-cycle is separated into its own two subplots because powering the drone off intentionally creates a very large age spike; if plotted on the same axis, it hides the behavior of the other runs.

The age plot answers: how old is the latest known state right now? The interarrival plot answers: while packets are arriving, how regular is the stream?


In [ ]:
telemetry_runs = ["E3-STATE-CLI-001", "E4-VIDEO-CLI-001", "E5-GUI-VID-IDLE-001", "E7-KBD-RESPONSE-001"]
power_run = "E8-PWR-CYCLE-001"
fig, axes = plt.subplots(4, 1, figsize=(11, 12), sharex=False)

def decorate_age_axis(ax, title):
    ax.axhspan(0, 200, color="#4C9F70", alpha=0.08, label="OK age <= 200 ms")
    ax.axhspan(200, 500, color="#E6A23C", alpha=0.08, label="DEGRADED age <= 500 ms")
    ax.axhline(500, color="#D95F5F", linestyle="--", linewidth=1, label="STALE above 500 ms")
    ax.set_ylabel("age (ms)")
    ax.set_title(title)

def decorate_interarrival_axis(ax, title):
    ax.axhspan(0, 300, color="#4C9F70", alpha=0.08, label="OK interarrival <= 300 ms")
    ax.axhspan(300, 500, color="#E6A23C", alpha=0.08, label="DEGRADED interarrival <= 500 ms")
    ax.axhline(500, color="#D95F5F", linestyle="--", linewidth=1, label="STALE above 500 ms")
    ax.set_ylabel("interarrival (ms)")
    ax.set_title(title)

for name in telemetry_runs:
    rows = get_rows(name)
    x_age, age = xy_series(rows, "telemetry_age_ms", nonnegative=True, max_s=150)
    x_int, inter = xy_series(rows, "telemetry_interarrival_ms", nonnegative=True, max_s=150)
    axes[0].plot(x_age, age, linewidth=1.4, label=experiment_label(name))
    axes[1].plot(x_int, inter, linewidth=1.4, label=experiment_label(name))

rows = get_rows(power_run)
x_age, age = xy_series(rows, "telemetry_age_ms", nonnegative=True, max_s=150)
x_int, inter = xy_series(rows, "telemetry_interarrival_ms", nonnegative=True, max_s=150)
axes[2].plot(x_age, age, linewidth=1.4, color="#4C78A8", label=experiment_label(power_run))
axes[3].plot(x_int, inter, linewidth=1.4, color="#4C78A8", label=experiment_label(power_run))

decorate_age_axis(axes[0], "Telemetry age over time, excluding power-cycle run")
decorate_interarrival_axis(axes[1], "Telemetry packet interarrival over time, excluding power-cycle run")
decorate_age_axis(axes[2], "E8 power-cycle telemetry age")
decorate_interarrival_axis(axes[3], "E8 power-cycle telemetry interarrival")
for ax in axes:
    ax.set_xlim(0, 150)
    ax.legend(fontsize=8, ncol=2)
axes[3].set_xlabel("elapsed time (s)")
savefig("telemetry_freshness_timeline.png")


## Video Pipeline Performance

The project uses FFmpeg Stream as the only video runtime path. Decode FPS is the main practical indicator of whether the video pipeline is keeping up with the stream. This comparison is capped at 150 seconds so longer runs do not dominate the visual scale.


In [ ]:
video_runs = [name for name in metrics if len(series(metrics[name], "decoder_fps_ema", nonzero=True)) > 0]
plt.figure(figsize=(10, 5))
for name in video_runs:
    rows = metrics[name]
    x, fps = xy_series(rows, "decoder_fps_ema", nonnegative=True, max_s=150)
    if len(fps):
        plt.plot(x, fps, linewidth=2, label=experiment_label(name))
plt.xlabel("elapsed time (s)")
plt.ylabel("decode FPS EMA")
plt.title("FFmpeg Stream decode FPS over time")
plt.legend()
savefig("video_decode_fps_comparison.png")


Video quality uses the same freshness concept as telemetry quality. It is not a subjective video sharpness score. A score of 100 means recent decoded video data is arriving within the freshness thresholds; 60 means delayed but present; 0 means stale or missing.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
add_quality_bands(ax)
for name in video_runs:
    rows = get_rows(name)
    x, q = xy_series(rows, "video_quality_score", nonnegative=True, max_s=150)
    if len(q):
        ax.step(x, q, where="post", linewidth=1.8, label=experiment_label(name))
ax.set_xlabel("elapsed time (s)")
ax.set_ylabel("video quality score")
ax.set_title("Video freshness quality over time")
ax.legend(fontsize=8, ncol=2)
savefig("video_quality_comparison.png")


## GUI Responsiveness

This comparison shows Qt timer delays over time. The reference lines are the intended timer periods: about 120 ms for the vision refresh timer and about 250 ms for the state/plot refresh timer.

A spike means the timer callback ran late. In practical terms, Qt wanted to execute a GUI update at the expected time, but the GUI event loop did not get CPU time until hundreds of milliseconds later. Possible causes include the GUI thread being busy painting/scaling frames, plot redraw work, blocking calls accidentally running on the GUI thread, OS scheduling pauses, graphics/driver stalls, or short contention with shared frame/state data.

The largest highlighted spikes occur near the moments when the blocking `takeoff` and `land` command calls returned with timeout and were logged. The plot also marks the telemetry-estimated physical takeoff/landing starts, which happen earlier than the timeout log rows.


In [ ]:
gui_runs = ["E5-GUI-VID-IDLE-001", "E7-KBD-RESPONSE-001"]
e7_state_rows = get_state("E7-KBD-RESPONSE-001-state")
e7_metric_rows = get_rows("E7-KBD-RESPONSE-001")
e7_takeoff_physical_s, e7_land_physical_s, _ = physical_flight_markers(e7_state_rows)
e7_takeoff_log_s = command_event_time(e7_metric_rows, "takeoff")
e7_land_log_s = command_event_time(e7_metric_rows, "land")

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=False)
summary_rows = []
spike_rows = []

def top_unique_spikes(x, y, limit=2):
    candidates = sorted(zip(x, y), key=lambda item: item[1], reverse=True)
    selected = []
    seen = set()
    for t, value in candidates:
        key = round(float(t), 1)
        if key in seen:
            continue
        seen.add(key)
        selected.append((float(t), float(value)))
        if len(selected) >= limit:
            break
    return selected

def annotate_gui_event_markers(ax):
    for t, label, color, style in [
        (e7_takeoff_physical_s, "physical takeoff start", "#4C9F70", ":"),
        (e7_takeoff_log_s, "takeoff timeout logged", "#2F7D4F", "--"),
        (e7_land_physical_s, "physical landing start", "#D95F5F", ":"),
        (e7_land_log_s, "land timeout logged", "#A33A3A", "--"),
    ]:
        if not math.isnan(t):
            ax.axvline(t, color=color, linestyle=style, linewidth=1.2, alpha=0.85)
            ax.text(t, 0.98, label, transform=ax.get_xaxis_transform(), rotation=90, va="top", ha="right", fontsize=8, color=color)

for name in gui_runs:
    rows = get_rows(name)
    label = experiment_label(name)
    x_v, vision = xy_series(rows, "gui_vision_tick_delay_ms", nonnegative=True)
    x_s, state = xy_series(rows, "gui_state_tick_delay_ms", nonnegative=True)
    axes[0].plot(x_v, vision, label=label, linewidth=1.6)
    axes[1].plot(x_s, state, label=label, linewidth=1.6)
    if name.startswith("E7"):
        for ax, x, y, timer in [(axes[0], x_v, vision, "vision"), (axes[1], x_s, state, "state")]:
            spikes = top_unique_spikes(x, y, limit=2)
            if spikes:
                sx = np.array([item[0] for item in spikes])
                sy = np.array([item[1] for item in spikes])
                ax.scatter(sx, sy, color="#D95F5F", zorder=5, s=28)
                for t, value in spikes:
                    spike_rows.append((timer, t, value))
    if len(vision):
        summary_rows.append((label, "vision", np.median(vision), np.percentile(vision, 95), np.max(vision)))
    if len(state):
        summary_rows.append((label, "state", np.median(state), np.percentile(state, 95), np.max(state)))

axes[0].axhline(120, color="#333333", linestyle="--", linewidth=1, label="expected vision tick ~120 ms")
axes[0].set_title("GUI vision timer delay over time")
axes[0].set_ylabel("delay (ms)")
axes[0].legend(fontsize=8)
axes[1].axhline(250, color="#333333", linestyle="--", linewidth=1, label="expected state tick ~250 ms")
axes[1].set_title("GUI state/plot timer delay over time")
axes[1].set_ylabel("delay (ms)")
axes[1].set_xlabel("elapsed time (s)")
axes[1].legend(fontsize=8)
for ax in axes:
    annotate_gui_event_markers(ax)
savefig("gui_tick_delay_idle_vs_flight.png")

print("GUI tick delay summary (ms):")
print("experiment | timer | median | p95 | max")
for label, timer, median, p95, max_v in summary_rows:
    print(f"{label} | {timer} | {median:.0f} | {p95:.0f} | {max_v:.0f}")
print("\nLargest highlighted E7 spikes:")
for timer, t, value in sorted(spike_rows, key=lambda item: item[2], reverse=True):
    print(f"{timer}: t={t:.1f}s delay={value:.0f}ms")
print(f"\nPhysical takeoff start: {e7_takeoff_physical_s:.1f}s; takeoff timeout logged: {e7_takeoff_log_s:.1f}s")
print(f"Physical landing start: {e7_land_physical_s:.1f}s; land timeout logged: {e7_land_log_s:.1f}s")


## State Recording And Thermal Context

State inter-sample gap is the elapsed time between two consecutive rows in the state recording CSV. It is a continuity metric for telemetry recording. If the receiver/logger is getting state packets regularly, the gaps stay close to the expected packet period. Large gaps mean one of three things happened: telemetry packets were not received, the receiver thread was delayed, or the recording path did not write a state sample on time.


In [ ]:
def plot_state_gaps(state_name, image_name, title):
    rows = get_state(state_name)
    t = series(rows, "steady_elapsed_ms", nonnegative=True)
    gaps = np.diff(t)
    gap_t = t[1:] / 1000.0
    plt.figure(figsize=(10, 4.8))
    plt.plot(gap_t, gaps, color="#4C78A8", linewidth=1)
    plt.axhline(300, color="#E6A23C", linestyle="--", label="300 ms")
    plt.axhline(500, color="#D95F5F", linestyle="--", label="500 ms")
    plt.xlabel("elapsed time (s)")
    plt.ylabel("inter-sample gap (ms)")
    plt.title(title)
    plt.legend()
    savefig(image_name)

plot_state_gaps("E5-GUI-VID-IDLE-001-state", "e5_state_gaps.png", "E5 state recording inter-sample gaps")


In [ ]:
plot_state_gaps("E7-KBD-RESPONSE-001-state", "e7_state_gaps.png", "E7 state recording inter-sample gaps")


Temperature is included because the Tello can become thermally constrained during long or repeated runs. `templ` and `temph` are the low/high internal temperature readings reported by the SDK.


In [ ]:
rows = get_state("E5-GUI-VID-IDLE-001-state")
x = series(rows, "steady_elapsed_ms", nonnegative=True) / 1000.0
plt.figure(figsize=(10, 5))
plt.plot(x, series(rows, "templ"), label="templ")
plt.plot(x, series(rows, "temph"), label="temph")
plt.xlabel("elapsed time (s)")
plt.ylabel("temperature (C)")
plt.title("E5 drone temperature during idle GUI/video run")
plt.legend()
savefig("e5_temperature.png")


## Keyboard Flight Operation

The keyboard experiment combines flight, telemetry, video, GUI rendering, and RC command generation. The next plot separates physical height from the vertical RC command so it is clear when the operator requested upward or downward motion.

The takeoff and landing markers in this plot are estimated from telemetry, not from the delayed `command:manual` log rows. The command rows are written after the blocking command call finishes, which can be several seconds late when the command times out. The physical takeoff marker is the first sustained increase in `h`/`tof`; the landing marker is the beginning of the final sustained descent toward the ground.


In [ ]:
state_rows = get_state("E7-KBD-RESPONSE-001-state")
takeoff_s, land_s, touchdown_s = physical_flight_markers(state_rows)

x = series(state_rows, "steady_elapsed_ms", nonnegative=True) / 1000.0
h = series(state_rows, "h")
tof = series(state_rows, "tof")
cmds_by_seq = OrderedDict()
for r in state_rows:
    seq = r.get("rc_sequence", "")
    if seq and seq not in cmds_by_seq:
        cmds_by_seq[seq] = r
cmds = list(cmds_by_seq.values())
rt = np.array([to_float(r.get("rc_steady_elapsed_ms")) / 1000.0 for r in cmds])
rc_c = np.array([to_float(r.get("rc_c"), 0) for r in cmds])

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(x[:len(h)], h, label="h: SDK height", color="#4C78A8")
axes[0].plot(x[:len(tof)], tof, label="tof: time-of-flight distance", color="#72B7B2")
axes[0].set_ylabel("height / tof (cm)")
axes[0].set_title("E7 height/tof and vertical RC command")
axes[0].legend()
axes[1].step(rt, rc_c, where="post", color="#E45756", label="rc_c: +up / -down")
axes[1].axhline(0, color="#333333", linewidth=0.8)
axes[1].set_ylabel("vertical RC command")
axes[1].set_xlabel("elapsed time (s)")
axes[1].legend()
for ax in axes:
    annotate_takeoff_land(ax, takeoff_s, land_s)
savefig("e7_height_and_vertical_rc.png")
print(f"physical takeoff start marker: {takeoff_s:.1f} s")
print(f"physical landing start marker: {land_s:.1f} s")
print(f"touchdown marker estimate: {touchdown_s:.1f} s")


The RC channel plot shows all four RC axes sent by the keyboard worker. Positive/negative values represent direction along each SDK control channel.


In [ ]:
plt.figure(figsize=(10, 5))
for col, label in [("rc_a", "left/right"), ("rc_b", "forward/back"), ("rc_c", "up/down"), ("rc_d", "yaw")]:
    vals = np.array([to_float(r.get(col), 0) for r in cmds])
    plt.step(rt, vals, where="post", label=label)
plt.xlabel("elapsed time (s)")
plt.ylabel("RC value")
plt.title("E7 RC channels sent by keyboard worker")
plt.legend()
savefig("e7_rc_channels.png")


The RC cadence plot focuses on the active worker cadence. The raw data contains one long inactive gap of about 21.7 seconds, which would stretch the histogram and hide the useful behavior. Therefore the histogram caps the displayed range at 250 ms and annotates the excluded long gap separately.


In [ ]:
gaps = np.diff(rt) * 1000.0 if len(rt) > 1 else np.array([])
active_gaps = gaps[gaps <= 250]
long_gaps = gaps[gaps > 250]
plt.figure(figsize=(9, 5))
if len(active_gaps):
    plt.hist(active_gaps, bins=np.arange(0, 260, 10), color="#4C78A8", edgecolor="white")
    median = np.median(active_gaps)
    p95 = np.percentile(active_gaps, 95)
    plt.axvline(median, color="#E45756", linestyle="--", label=f"median={median:.0f} ms")
    plt.axvline(p95, color="#4C9F70", linestyle=":", label=f"p95={p95:.0f} ms")
plt.xlabel("RC send gap during active cadence (ms)")
plt.ylabel("count")
plt.title("E7 keyboard RC worker send cadence")
if len(long_gaps):
    plt.text(0.98, 0.92, f"excluded idle gaps >250 ms: {len(long_gaps)}\nmax idle gap: {np.max(long_gaps)/1000:.1f} s", transform=plt.gca().transAxes, ha="right", va="top", bbox=dict(facecolor="white", alpha=0.85, edgecolor="#999"))
plt.legend()
savefig("e7_rc_gap_histogram.png")


The operation quality plot combines telemetry/video freshness and decode FPS during keyboard flight. Quality score again means freshness/continuity: 100 is OK, 60 is degraded, and 0 is stale or no data.


In [ ]:
rows = get_rows("E7-KBD-RESPONSE-001")
state_rows = get_state("E7-KBD-RESPONSE-001-state")
takeoff_s, land_s, _ = physical_flight_markers(state_rows)
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
add_quality_bands(axes[0])
x_t, tq = xy_series(rows, "telemetry_quality_score", nonnegative=True)
x_v, vq = xy_series(rows, "video_quality_score", nonnegative=True)
axes[0].step(x_t, tq, where="post", label="telemetry quality", color="#4C9F70", linestyle="--")
axes[0].step(x_v, vq, where="post", label="video quality", color="#4C78A8")
axes[0].set_ylabel("quality score")
axes[0].legend(fontsize=8)
axes[0].set_title("E7 telemetry/video freshness during keyboard flight")
x_fps, fps = xy_series(rows, "decoder_fps_ema", nonnegative=True)
axes[1].plot(x_fps, fps, color="#F28E2B", label="decode FPS")
axes[1].set_ylabel("decode FPS EMA")
axes[1].set_xlabel("elapsed time (s)")
axes[1].legend()
for ax in axes:
    annotate_takeoff_land(ax, takeoff_s, land_s)
savefig("e7_operation_quality_video.png")


### RC Reaction Latency Estimate

The reaction estimate is computed from vertical RC pulses (`rc_c != 0`). For each pulse, the script finds the first later telemetry sample where `h` or `tof` changes by at least 5 cm in the expected direction. Blue points are upward commands; orange points are downward commands. A 500 ms tolerance after the pulse end is allowed because telemetry samples are asynchronous with RC command timestamps.


In [ ]:
def unique_rc_commands(state_rows):
    seen = OrderedDict()
    for r in state_rows:
        seq = r.get("rc_sequence", "")
        if seq == "":
            continue
        if seq not in seen:
            seen[seq] = r
    cmds = list(seen.values())
    cmds.sort(key=lambda r: to_float(r.get("rc_steady_elapsed_ms"), 0))
    return cmds


def estimate_vertical_reactions(state_rows, threshold_cm=5.0, end_tolerance_ms=500.0):
    cmds = unique_rc_commands(state_rows)
    samples = []
    for r in state_rows:
        t = to_float(r.get("steady_elapsed_ms"))
        if math.isnan(t):
            continue
        samples.append({"t": t, "h": to_float(r.get("h")), "tof": to_float(r.get("tof"))})

    pulses = []
    active = None
    for cmd in cmds:
        rc_c_value = to_float(cmd.get("rc_c"), 0)
        t = to_float(cmd.get("rc_steady_elapsed_ms"))
        if math.isnan(t):
            continue
        sign = 1 if rc_c_value > 0 else (-1 if rc_c_value < 0 else 0)
        if sign == 0:
            if active is not None:
                active["end_t"] = t
                pulses.append(active)
                active = None
            continue
        if active is None or active["sign"] != sign:
            if active is not None:
                active["end_t"] = t
                pulses.append(active)
            active = {"start_t": t, "sign": sign, "rc_c": rc_c_value}
    if active is not None:
        active["end_t"] = active["start_t"] + 2000
        pulses.append(active)

    reactions = []
    for pulse in pulses:
        start_t = pulse["start_t"]
        sign = pulse["sign"]
        baseline = next((s for s in samples if s["t"] >= start_t), None)
        if baseline is None:
            continue
        base_h = baseline["h"]
        base_tof = baseline["tof"]
        response_t = None
        for sample in samples:
            if sample["t"] <= start_t:
                continue
            if sample["t"] > pulse["end_t"] + end_tolerance_ms:
                break
            dh = sample["h"] - base_h if not (math.isnan(sample["h"]) or math.isnan(base_h)) else np.nan
            dtof = sample["tof"] - base_tof if not (math.isnan(sample["tof"]) or math.isnan(base_tof)) else np.nan
            moved_h = not math.isnan(dh) and sign * dh >= threshold_cm
            moved_tof = not math.isnan(dtof) and sign * dtof >= threshold_cm
            if moved_h or moved_tof:
                response_t = sample["t"]
                break
        if response_t is not None:
            reactions.append({"start_s": start_t / 1000.0, "latency_ms": response_t - start_t, "rc_c": pulse["rc_c"]})
    return reactions

state_rows = get_state("E7-KBD-RESPONSE-001-state")
reactions = estimate_vertical_reactions(state_rows)
latencies = np.array([r["latency_ms"] for r in reactions])
colors = ["#4C78A8" if r["rc_c"] > 0 else "#F28E2B" for r in reactions]

plt.figure(figsize=(10, 5))
plt.scatter([r["start_s"] for r in reactions], latencies, c=colors, s=55, edgecolor="white")
if len(latencies):
    plt.axhline(np.median(latencies), color="#D95F5F", linestyle="--", label=f"median={np.median(latencies):.0f} ms")
    plt.axhline(np.mean(latencies), color="#4C9F70", linestyle=":", label=f"mean={np.mean(latencies):.0f} ms")
plt.scatter([], [], c="#4C78A8", label="upward rc_c > 0")
plt.scatter([], [], c="#F28E2B", label="downward rc_c < 0")
plt.xlabel("RC pulse start time (s)")
plt.ylabel("estimated reaction latency (ms)")
plt.title("E7 estimated vertical RC reaction latency")
plt.legend()
savefig("e7_rc_reaction_latency.png")

if len(latencies):
    print(f"valid vertical pulses: {len(latencies)}")
    print(f"median latency: {np.median(latencies):.0f} ms")
    print(f"mean latency: {np.mean(latencies):.0f} ms")
    print(f"range: {np.min(latencies):.0f} to {np.max(latencies):.0f} ms")


### E7 Drone State Subplot Matrix

This matrix shows the main state variables around the real flight window only: from about 7 seconds before the telemetry-estimated takeoff start to about 7 seconds after the telemetry-estimated landing start. This keeps the physical flight context visible without spending plot space on the full idle pre/post-run data.


In [ ]:
rows = get_state("E7-KBD-RESPONSE-001-state")
takeoff_s, land_s, touchdown_s = physical_flight_markers(rows)
window_start = max(0, takeoff_s - 7) if not math.isnan(takeoff_s) else 0
window_end = land_s + 7 if not math.isnan(land_s) else max(elapsed_seconds(rows))

x_all = series(rows, "steady_elapsed_ms", nonnegative=True) / 1000.0
state_cols = [
    ("pitch", "deg"), ("roll", "deg"), ("yaw", "deg"),
    ("vgx", "cm/s"), ("vgy", "cm/s"), ("vgz", "cm/s"),
    ("tof", "cm"), ("h", "cm"), ("baro", "cm"),
    ("bat", "%"), ("templ", "deg C"), ("temph", "deg C"),
    ("agx", "0.001g"), ("agy", "0.001g"), ("agz", "0.001g"),
]
fig, axes = plt.subplots(5, 3, figsize=(14, 12), sharex=True)
axes = axes.ravel()
for ax, (col, unit) in zip(axes, state_cols):
    y = series(rows, col)
    n = min(len(x_all), len(y))
    xm = x_all[:n]
    ym = y[:n]
    local_mask = (xm >= window_start) & (xm <= window_end)
    ax.plot(xm[local_mask], ym[local_mask], linewidth=1.2)
    annotate_takeoff_land(ax, takeoff_s, land_s)
    ax.set_title(col)
    ax.set_ylabel(unit)
    ax.grid(True, alpha=0.25)
for ax in axes[len(state_cols):]:
    ax.axis("off")
for ax in axes[-3:]:
    ax.set_xlabel("elapsed time (s)")
fig.suptitle("E7 drone state evolution around physical takeoff/landing", y=1.01)
savefig("e7_drone_state_subplots.png")
print(f"state subplot window: {window_start:.1f}s to {window_end:.1f}s")
print(f"physical takeoff marker: {takeoff_s:.1f}s; physical landing marker: {land_s:.1f}s; touchdown estimate: {touchdown_s:.1f}s")


## Recovery Evidence

The recovery plots show how freshness and connection-state indicators behaved during power-cycle and Wi-Fi-loss experiments. The quality score bands are the same as above: 100 means OK, 60 means degraded, and 0 means stale or no data.


In [ ]:
rows = get_rows("E8-PWR-CYCLE-001")
x_t, tq = xy_series(rows, "telemetry_quality_score", nonnegative=True)
x_v, vq = xy_series(rows, "video_quality_score", nonnegative=True)
fig, ax = plt.subplots(figsize=(10, 5))
add_quality_bands(ax)
ax.step(x_t, tq, where="post", label="telemetry quality", color="#4C9F70", linestyle="--", linewidth=2.0)
ax.step(x_v, vq, where="post", label="video quality", color="#4C78A8", linestyle="-", linewidth=1.6)
recovery_x = [to_float(r.get("elapsed_ms")) / 1000.0 for r in rows if to_int(r.get("recovery_attempted"), 0) == 1]
for rx in recovery_x:
    ax.axvline(rx, color="#D95F5F", alpha=0.25)
ax.set_xlabel("elapsed time (s)")
ax.set_ylabel("quality score")
ax.set_title("E8 power-cycle recovery quality timeline")
ax.legend(fontsize=8, ncol=2)
savefig("e8_power_cycle_quality.png")


In [ ]:
rows = get_rows("E8-PWR-CYCLE-001")
tx, ty = xy_series(rows, "telemetry_age_ms", nonnegative=True)
vx, vy = xy_series(rows, "video_age_ms", nonnegative=True)
plt.figure(figsize=(10, 5))
plt.plot(tx, ty, label="telemetry age", color="#4C9F70")
plt.plot(vx, vy, label="video age", color="#4C78A8")
plt.axhline(500, color="#D95F5F", linestyle="--", label="500 ms stale threshold")
plt.xlabel("elapsed time (s)")
plt.ylabel("packet/frame age (ms)")
plt.title("E8 power-cycle data age")
plt.legend()
savefig("e8_power_cycle_age.png")


In [ ]:
rows = get_rows("E9-WIFI-LOSS-001")
x_fail, failures = xy_series(rows, "command_failures", nonnegative=True)
x_lat, latency = xy_series(rows, "command_latency_ms_avg", nonnegative=True)
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(x_fail, failures, color="#D95F5F", label="command failures")
ax1.set_ylabel("command failures")
ax1.set_xlabel("elapsed time (s)")
ax2 = ax1.twinx()
ax2.plot(x_lat, latency, color="#4C78A8", label="avg command latency")
ax2.set_ylabel("avg command latency (ms)")
fig.suptitle("E9 Wi-Fi loss/reconnect command behavior")
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc="upper left")
savefig("e9_wifi_loss_reconnect.png")


## Reproducibility Check

This final cell lists the generated image files. If a file is missing, rerun the notebook from the top.


In [ ]:
expected = [
    "experiment_data_availability.png",
    "e2_command_latency.png",
    "telemetry_age_comparison.png",
    "telemetry_freshness_timeline.png",
    "video_decode_fps_comparison.png",
    "video_quality_comparison.png",
    "gui_tick_delay_idle_vs_flight.png",
    "e5_state_gaps.png",
    "e7_state_gaps.png",
    "e5_temperature.png",
    "e7_height_and_vertical_rc.png",
    "e7_rc_channels.png",
    "e7_rc_gap_histogram.png",
    "e7_operation_quality_video.png",
    "e7_rc_reaction_latency.png",
    "e7_drone_state_subplots.png",
    "e8_power_cycle_quality.png",
    "e8_power_cycle_age.png",
    "e9_wifi_loss_reconnect.png",
]
for name in expected:
    path = IMAGES_DIR / name
    print(f"{'OK' if path.exists() else 'MISSING'} {name}")
